# CIFAR-100 / SVHN — Bootstrap CI 계산

**목적:** 표 6의 CIFAR-100(현재 ~1.000 근삿값)과 SVHN 행에 정확한 AUROC ± 95% CI를 채운다.  
**프로토콜:** bootstrap n=1,000, seed=42 (CIFAR-10 / TinyImageNet / ImageNet과 동일)  
**참조 노트북:** `scan_finetune_cifar100_svhn.ipynb` (데이터 경로·모델 경로 그대로 사용)

## 0. 경로 설정

> 수정이 필요한 항목은 `DECODER_PATH`와 `SCAN_ROOT` 
> 데이터·모델 경로는 `scan_finetune_cifar100_svhn.ipynb`와 동일

In [ ]:
# ── 수정 필요: SCAN 관련 경로 ─────────────────────────────────────
SCAN_ROOT    = '.'                     # SCAN 패키지 루트
DECODER_PATH = './scan_decoder_resnet50_imagenet.pth'  # ImageNet SCAN 디코더

# ── 수정 불필요: finetune 노트북과 동일한 경로 ───────────────────
MODEL_C100   = './resnet50_cifar100_finetuned.pt'
MODEL_SVHN   = './resnet50_svhn_finetuned.pt'
DATA_C100    = './cifar_data'
DATA_SVHN    = './svhn_data'
RESULTS_DIR  = './ci_results_c100_svhn'

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
print('경로 설정 완료')
print(f'  CIFAR-100 모델: {MODEL_C100}  존재={os.path.exists(MODEL_C100)}')
print(f'  SVHN    모델:   {MODEL_SVHN}  존재={os.path.exists(MODEL_SVHN)}')
print(f'  DECODER:        {DECODER_PATH}  존재={os.path.exists(DECODER_PATH)}')

## 1. Import 및 공통 유틸리티

In [ ]:
import sys, json, random, io
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import torchvision.models as models
from torchvision.models import ResNet50_Weights
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from PIL import Image, ImageFilter
import torchvision.transforms.functional as TF
from tqdm import tqdm

sys.path.insert(0, SCAN_ROOT)
from scan import SCAN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# 재현성 고정
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# ── Bootstrap CI ──────────────────────────────────────────────────
def bootstrap_auroc_ci(y_true, y_score, n_boot=1000, seed=42):
    """95% CI, bootstrap n=1,000, seed=42 — 기존 CIFAR-10 프로토콜과 동일"""
    rng = np.random.RandomState(seed)
    n   = len(y_true)
    aucs = []
    for _ in range(n_boot):
        idx = rng.choice(n, n, replace=True)
        yt, ys = y_true[idx], y_score[idx]
        if len(np.unique(yt)) < 2:
            continue
        aucs.append(roc_auc_score(yt, ys))
    aucs = np.array(aucs)
    return float(np.mean(aucs)), float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5))

def fmt(mean, lo, hi):
    """논문 표 형식: x.xxx±x.xxx"""
    return f'{mean:.3f}±{(hi-lo)/2:.3f}'

print('Bootstrap CI 함수 정의 완료')

## 2. 모델 래퍼 — 정규화를 모델 내부에서 처리

공격 코드와 특징 추출 코드가 항상 **raw [0,1] 이미지**를 받도록,  
정규화를 모델 내부 `forward()`에서 처리하는 래퍼 클래스를 사용합니다.

In [ ]:
class NormModel(nn.Module):
    """raw [0,1] 텐서를 받아 내부에서 정규화 후 분류기에 통과"""
    def __init__(self, backbone, mean, std):
        super().__init__()
        self.backbone = backbone
        self.register_buffer('mean', torch.tensor(mean, dtype=torch.float32).view(1,3,1,1))
        self.register_buffer('std',  torch.tensor(std,  dtype=torch.float32).view(1,3,1,1))

    def forward(self, x):
        return self.backbone((x - self.mean) / self.std)

def load_resnet50(ckpt_path, num_classes, mean, std, device):
    """finetune 노트북과 동일한 checkpoint 형식 로드"""
    backbone = models.resnet50(weights=None)
    backbone.fc = nn.Linear(2048, num_classes)
    ckpt = torch.load(ckpt_path, map_location=device)
    # finetune 노트북 저장 형식: {'state_dict': ..., 'val_acc': ..., 'num_classes': ...}
    state = ckpt.get('state_dict', ckpt)
    state = {k.replace('module.', ''): v for k, v in state.items()}  # DataParallel prefix 제거
    backbone.load_state_dict(state, strict=True)
    backbone.eval()
    model = NormModel(backbone, mean, std).to(device).eval()
    return model

print('모델 래퍼 정의 완료')

## 3. 탐지 특징 함수

In [ ]:
def _gauss_kernel(sigma, device):
    ks = max(int(6 * sigma + 1) | 1, 3)  # 홀수 보장
    coords = torch.arange(ks, dtype=torch.float32, device=device) - ks // 2
    g = torch.exp(-coords**2 / (2 * sigma**2))
    g /= g.sum()
    g2d = (g[:, None] * g[None, :]).unsqueeze(0).unsqueeze(0)  # (1,1,k,k)
    return g2d.repeat(3, 1, 1, 1), ks // 2  # (3,1,k,k), padding

def gauss_blur(x, sigma=0.5):
    k, pad = _gauss_kernel(sigma, x.device)
    return F.conv2d(x, k, padding=pad, groups=3)

# ── HF-Energy ─────────────────────────────────────────────────────
def hfe(x, sigma=0.5):
    """mean|x - GaussBlur(x)| / 255, x in [0,1]"""
    return (x - gauss_blur(x, sigma)).abs().mean(dim=[1,2,3]).cpu().numpy()

# ── GaussianL1 ────────────────────────────────────────────────────
@torch.no_grad()
def gaussian_l1(x, model, sigma=0.5):
    """||p(x) - p(GaussBlur(x))||_1"""
    blurred = gauss_blur(x, sigma)
    p1 = torch.softmax(model(x), 1)
    p2 = torch.softmax(model(blurred), 1)
    return (p1 - p2).abs().sum(1).cpu().numpy()

# ── PredL1 (JPEG + Median3) ───────────────────────────────────────
def _jpeg_median_batch(x):
    q = random.choice([65, 75, 85])
    out = []
    for img in x:
        pil = TF.to_pil_image(img.cpu().clamp(0,1))
        buf = io.BytesIO()
        pil.save(buf, 'JPEG', quality=q)
        buf.seek(0)
        pil2 = Image.open(buf).convert('RGB').filter(ImageFilter.MedianFilter(3))
        out.append(TF.to_tensor(pil2))
    return torch.stack(out).to(x.device)

@torch.no_grad()
def pred_l1(x, model):
    sq = _jpeg_median_batch(x)
    p1 = torch.softmax(model(x), 1)
    p2 = torch.softmax(model(sq), 1)
    return (p1 - p2).abs().sum(1).cpu().numpy()

# ── Spatial-HF-SCAN ───────────────────────────────────────────────
def spatial_hf_scan(x, scanner, sigma=0.5):
    """mean|C(x) - C(GaussBlur(x))|, x in [0,1], SCAN expects [0,255]"""
    x_255      = (x * 255).clamp(0, 255)
    blurred_255 = (gauss_blur(x, sigma) * 255).clamp(0, 255)
    with torch.no_grad():
        c_orig, _ = scanner(x_255, percentile=70)
        c_blur, _ = scanner(blurred_255, percentile=70)
    # c shape: (B, H, W)
    return (c_orig - c_blur).abs().mean(dim=[1,2]).cpu().numpy()

print('탐지 특징 함수 정의 완료')

## 4. 공격 함수

In [ ]:
def fgsm(model, x, y, eps=8/255):
    xc = x.clone().detach().requires_grad_(True)
    nn.CrossEntropyLoss()(model(xc), y).backward()
    return (x + eps * xc.grad.sign()).clamp(0, 1).detach()

def pgd(model, x, y, eps=8/255, alpha=2/255, steps=20):
    x_adv = (x + torch.empty_like(x).uniform_(-eps, eps)).clamp(0,1).detach()
    for _ in range(steps):
        x_adv.requires_grad_(True)
        nn.CrossEntropyLoss()(model(x_adv), y).backward()
        with torch.no_grad():
            x_adv = (x_adv + alpha * x_adv.grad.sign())
            x_adv = torch.max(torch.min(x_adv, x + eps), x - eps).clamp(0, 1)
    return x_adv.detach()

def cw(model, x, y, c=1.0, kappa=0, steps=100, lr=0.01):
    """C&W L2, atanh 초기화 (회색 이미지 출발점 방지)"""
    w = torch.atanh((2*x - 1).clamp(-1+1e-6, 1-1e-6)).detach().clone().requires_grad_(True)
    opt = torch.optim.Adam([w], lr=lr)
    best = x.clone()
    best_l2 = torch.full((x.shape[0],), 1e9, device=x.device)
    for _ in range(steps):
        xa = ((torch.tanh(w) + 1) / 2).clamp(0, 1)
        l2 = ((xa - x)**2).sum(dim=[1,2,3]).sqrt()
        logits = model(xa)
        real  = logits.gather(1, y.unsqueeze(1)).squeeze(1)
        other = (logits - 1e4 * F.one_hot(y, logits.size(1)).float()).max(1)[0]
        loss = (l2 + c * (real - other + kappa).clamp(min=0)).sum()
        opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            xa_d = ((torch.tanh(w) + 1) / 2).clamp(0, 1)
            l2_d = ((xa_d - x)**2).sum(dim=[1,2,3]).sqrt()
            ok   = (model(xa_d).argmax(1) != y) & (l2_d < best_l2)
            best[ok] = xa_d[ok]; best_l2[ok] = l2_d[ok]
    return best.detach()

print('공격 함수 정의 완료')

## 5. 공통 실험 실행 함수

In [ ]:
def run_ci_experiment(name, model, scanner, loader, n_samples=500, bs=32):
    """
    Parameters
    ----------
    name     : 데이터셋 이름 (로그용)
    model    : NormModel (raw [0,1] 입력)
    scanner  : SCAN 객체
    loader   : DataLoader (ToTensor만, 정규화 없음)
    n_samples: 정상/공격 샘플 각 수
    """
    model.eval()

    # ── 1. 정상 샘플 수집 (올바르게 분류된 것만) ─────────────────
    print(f'[{name}] 정상 샘플 수집 ...')
    imgs_all, lbls_all = [], []
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        with torch.no_grad():
            ok = model(imgs).argmax(1) == lbls
        imgs_all.append(imgs[ok]); lbls_all.append(lbls[ok])
        if sum(t.shape[0] for t in imgs_all) >= n_samples * 2:
            break
    imgs_all = torch.cat(imgs_all)[:n_samples * 2]
    lbls_all = torch.cat(lbls_all)[:n_samples * 2]

    calib_x, calib_y = imgs_all[:n_samples],  lbls_all[:n_samples]
    test_x,  test_y  = imgs_all[n_samples:],  lbls_all[n_samples:]
    print(f'  calib={len(calib_x)}, test={len(test_x)}')

    # ── 2. 특징 추출 helper ───────────────────────────────────────
    def extract(x_batch):
        H, G, P, S = [], [], [], []
        for i in range(0, len(x_batch), bs):
            b = x_batch[i:i+bs]
            H.append(hfe(b))
            G.append(gaussian_l1(b, model))
            P.append(pred_l1(b, model))
            S.append(spatial_hf_scan(b, scanner))
        return (np.concatenate(H), np.concatenate(G),
                np.concatenate(P), np.concatenate(S))

    # ── 3. 캘리브레이션: μ, σ 추정 ──────────────────────────────
    print(f'[{name}] 캘리브레이션 ...')
    calib_feat = extract(calib_x)
    mu = [f.mean() for f in calib_feat]
    si = [f.std() + 1e-8 for f in calib_feat]

    def score(feats):
        """HFE 단독 z-score (CIFAR 최적) and 앙상블 z-score"""
        zs = [(f - m) / s for f, m, s in zip(feats, mu, si)]
        z_hfe = np.abs(zs[0])          # HFE 단독
        z_ens = np.abs(np.stack(zs, 1).mean(1))  # 앙상블
        return z_hfe, z_ens

    # ── 4. 테스트 정상 점수 ───────────────────────────────────────
    print(f'[{name}] 테스트 정상 특징 추출 ...')
    clean_feat = extract(test_x)
    clean_hfe_sc, clean_ens_sc = score(clean_feat)

    # ── 5. 공격별 실험 ────────────────────────────────────────────
    atk_fns = {
        'FGSM': lambda x, y: fgsm(model, x, y),
        'PGD':  lambda x, y: pgd(model, x, y),
        'CW':   lambda x, y: cw(model, x, y),
    }
    results = {}
    adv_hfe_all, adv_ens_all = [], []

    for atk_name, atk_fn in atk_fns.items():
        print(f'[{name}] {atk_name} 공격 생성 ...')
        adv_list = []
        for i in tqdm(range(0, min(len(test_x), n_samples), bs), desc=atk_name):
            bx = test_x[i:i+bs]; by = test_y[i:i+bs]
            xa = atk_fn(bx, by)
            with torch.no_grad():
                ok = model(xa).argmax(1) != by
            adv_list.append(xa[ok])
        adv = torch.cat(adv_list)
        print(f'  성공 샘플: {len(adv)}/{n_samples}')

        adv_feat = extract(adv)
        adv_hfe_sc, adv_ens_sc = score(adv_feat)

        n_use = min(len(adv_hfe_sc), len(clean_hfe_sc))
        yt = np.concatenate([np.zeros(n_use), np.ones(n_use)])

        auc_h, lo_h, hi_h = bootstrap_auroc_ci(
            yt, np.concatenate([clean_hfe_sc[:n_use], adv_hfe_sc[:n_use]]))
        auc_e, lo_e, hi_e = bootstrap_auroc_ci(
            yt, np.concatenate([clean_ens_sc[:n_use], adv_ens_sc[:n_use]]))

        results[atk_name] = dict(
            n_success=len(adv),
            hfe=dict(auc=auc_h, lo=lo_h, hi=hi_h),
            ens=dict(auc=auc_e, lo=lo_e, hi=hi_e),
        )
        print(f'  HFE: {fmt(auc_h, lo_h, hi_h)}   Ens: {fmt(auc_e, lo_e, hi_e)}')

        adv_hfe_all.append(adv_hfe_sc[:n_use])
        adv_ens_all.append(adv_ens_sc[:n_use])

    # ── 6. Overall ────────────────────────────────────────────────
    all_adv_h = np.concatenate(adv_hfe_all)
    all_adv_e = np.concatenate(adv_ens_all)
    n_tot = len(all_adv_h)
    clean_rep_h = np.tile(clean_hfe_sc, 3)[:n_tot]
    clean_rep_e = np.tile(clean_ens_sc, 3)[:n_tot]
    yt_all = np.concatenate([np.zeros(n_tot), np.ones(n_tot)])

    auc_oh, lo_oh, hi_oh = bootstrap_auroc_ci(
        yt_all, np.concatenate([clean_rep_h, all_adv_h]))
    auc_oe, lo_oe, hi_oe = bootstrap_auroc_ci(
        yt_all, np.concatenate([clean_rep_e, all_adv_e]))

    results['Overall'] = dict(
        hfe=dict(auc=auc_oh, lo=lo_oh, hi=hi_oh),
        ens=dict(auc=auc_oe, lo=lo_oe, hi=hi_oe),
    )
    print(f'[{name}] Overall  HFE: {fmt(auc_oh, lo_oh, hi_oh)}   Ens: {fmt(auc_oe, lo_oe, hi_oe)}')

    # JSON 저장
    path = os.path.join(RESULTS_DIR, f'{name.lower().replace("-","")}_ci.json')
    with open(path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'  JSON 저장: {path}')
    return results

print('실험 함수 정의 완료')

## 6. CIFAR-100 실험

In [ ]:
# ── CIFAR-100 정규화 (finetune 노트북과 동일) ─────────────────────
C100_MEAN = [0.5071, 0.4867, 0.4408]
C100_STD  = [0.2675, 0.2565, 0.2761]

# 모델 로드
model_c100 = load_resnet50(MODEL_C100, num_classes=100,
                            mean=C100_MEAN, std=C100_STD, device=device)
print(f'CIFAR-100 모델 로드 완료  (val_acc={torch.load(MODEL_C100)["val_acc"]:.4f})')

In [ ]:
# ── CIFAR-100 SCAN 설정 ───────────────────────────────────────────
# preprocess: [0,255] → 정규화된 텐서 (SCAN 내부에서 호출)
def preprocess_c100(x_255):
    """SCAN이 호출하는 전처리: 0-255 텐서 → 정규화"""
    x = x_255 / 255.0
    mean = torch.tensor(C100_MEAN, device=x.device).view(1,3,1,1)
    std  = torch.tensor(C100_STD,  device=x.device).view(1,3,1,1)
    return (x - mean) / std

scanner_c100 = SCAN(
    target_model=model_c100.backbone,  # NormModel 내부 백본
    target_layer='layer4',
    image_size=(224, 224),
    use_gradient_mask=True,
    device=device,
    num_classes=100
)
scanner_c100.set_preprocess(preprocess_c100)
scanner_c100.load_decoder(DECODER_PATH)
print('CIFAR-100 SCAN 설정 완료')

In [ ]:
# ── CIFAR-100 데이터로더 (ToTensor만, 정규화 없음) ────────────────
# finetune 노트북의 _tf_val과 Normalize 제외 버전
tf_c100 = T.Compose([T.Resize(224), T.ToTensor()])

ds_c100 = torchvision.datasets.CIFAR100(
    DATA_C100, train=False, download=False, transform=tf_c100)
ld_c100 = DataLoader(ds_c100, batch_size=64, shuffle=True,
                      num_workers=2, pin_memory=True)
print(f'CIFAR-100 테스트셋: {len(ds_c100):,} 샘플')

In [ ]:
# ── 실험 실행 ─────────────────────────────────────────────────────
res_c100 = run_ci_experiment(
    name='CIFAR-100',
    model=model_c100,
    scanner=scanner_c100,
    loader=ld_c100,
    n_samples=500,
    bs=32
)

## 7. SVHN 실험

In [ ]:
# ── SVHN 정규화 (finetune 노트북과 동일) ──────────────────────────
SVHN_MEAN = [0.4377, 0.4438, 0.4728]
SVHN_STD  = [0.1980, 0.2010, 0.1970]

model_svhn = load_resnet50(MODEL_SVHN, num_classes=10,
                            mean=SVHN_MEAN, std=SVHN_STD, device=device)
print(f'SVHN 모델 로드 완료  (val_acc={torch.load(MODEL_SVHN)["val_acc"]:.4f})')

In [ ]:
# ── SVHN SCAN 설정 ────────────────────────────────────────────────
def preprocess_svhn(x_255):
    x = x_255 / 255.0
    mean = torch.tensor(SVHN_MEAN, device=x.device).view(1,3,1,1)
    std  = torch.tensor(SVHN_STD,  device=x.device).view(1,3,1,1)
    return (x - mean) / std

scanner_svhn = SCAN(
    target_model=model_svhn.backbone,
    target_layer='layer4',
    image_size=(224, 224),
    use_gradient_mask=True,
    device=device,
    num_classes=10
)
scanner_svhn.set_preprocess(preprocess_svhn)
scanner_svhn.load_decoder(DECODER_PATH)
print('SVHN SCAN 설정 완료')

In [ ]:
# ── SVHN 데이터로더 (ToTensor만) ──────────────────────────────────
tf_svhn = T.Compose([T.Resize(224), T.ToTensor()])

ds_svhn = torchvision.datasets.SVHN(
    DATA_SVHN, split='test', download=False, transform=tf_svhn)
ld_svhn = DataLoader(ds_svhn, batch_size=64, shuffle=True,
                      num_workers=2, pin_memory=True)
print(f'SVHN 테스트셋: {len(ds_svhn):,} 샘플')

In [ ]:
res_svhn = run_ci_experiment(
    name='SVHN',
    model=model_svhn,
    scanner=scanner_svhn,
    loader=ld_svhn,
    n_samples=500,
    bs=32
)

## 8. 최종 결과 — 표 6에 삽입할 값 출력

In [ ]:
print('\n' + '='*70)
print('표 6. 최종 결과 — AUROC ± 95% CI (bootstrap n=1,000)')
print('(HFE 단일 = CIFAR 최적 앙상블)')
print('='*70)
print(f'{"데이터셋":15s} | {"FGSM":12s} | {"PGD":12s} | {"C&W":12s} | {"Overall":12s}')
print('-'*70)

for dname, res in [('CIFAR-100', res_c100), ('SVHN', res_svhn)]:
    row_hfe = []
    row_ens = []
    for atk in ['FGSM', 'PGD', 'CW', 'Overall']:
        r = res[atk]
        row_hfe.append(fmt(r['hfe']['auc'], r['hfe']['lo'], r['hfe']['hi']))
        row_ens.append(fmt(r['ens']['auc'], r['ens']['lo'], r['ens']['hi']))
    print(f'{dname+" (HFE)":15s} | {" | ".join(row_hfe)}')
    print(f'{dname+" (Ens)":15s} | {" | ".join(row_ens)}')
    print()

print()
print('── Markdown 표 형식 (논문에 바로 붙여넣기) ─────────────────────')
print('| 데이터셋 | FGSM | PGD | C&W | Overall |')
print('|---------|:----:|:---:|:---:|:-------:|')
for dname, res in [('CIFAR-100', res_c100), ('SVHN', res_svhn)]:
    vals = []
    for atk in ['FGSM', 'PGD', 'CW', 'Overall']:
        r = res[atk]
        vals.append(fmt(r['hfe']['auc'], r['hfe']['lo'], r['hfe']['hi']))
    print(f'| {dname} | {" | ".join(vals)} |')